Thyrosim Sweep Notebook

For testing and running main Thyrosim model by sweeping across a large combination of input parameters 
Inputs: Sex, Height, Weight, LT3, LT4, RTF
Outputs: FT4, FT3, TT3, TSH

Note: This code is not used since it's quite slow. Please see "sweep.py" instead

In [1]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from itertools import product
import matplotlib.pyplot as plt
from thyrosim_model import simulate_patient
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm

In [ ]:
# Basic implementation
def compute_means(df, window=50):
    tail = df.tail(window)

    return {
        "FT4_mean": tail["FT4"].mean(),
        "FT3_mean": tail["FT3"].mean(),
        "TT3_mean": tail["TT3"].mean(),
        "TSH_mean": tail["TSH"].mean(),
    }


def generate_full_dataset():
    # Full sweep
    #heights = list(range(150, 185, 5))
    #weights = list(range(50, 75, 5))
    #sexes = ["male", "female"]
    #lt4_vals = list(range(25, 50))
    #lt3_vals = list(range(5, 10))
    #rtf_vals = np.linspace(0.0, 1.0, 10)

    # test sweep
    heights = [150, 180]
    weights = [50, 70]
    sexes = ["male", "female"]
    lt4_vals = [25, 50]
    lt3_vals = [5, 10]
    rtf_vals = np.linspace(0.0, 1.0, 2)

    # total number of runs
    total = (
        len(heights)
        * len(weights)
        * len(sexes)
        * len(lt4_vals)
        * len(lt3_vals)
        * len(rtf_vals)
    )

    print(f"Total simulations: {total}")

    rows = []
    count = 0

    for h, w, s, lt4, lt3, rtf in product(
        heights, weights, sexes, lt4_vals, lt3_vals, rtf_vals
    ):
        count += 1

        # percentage progress
        progress = (count / total) * 100

        if count % 50 == 0 or count == 1:
            print(f"[{count}/{total}] ({progress:.2f}%)")

        df = simulate_patient(
            height=h,
            weight=w,
            sex=s,
            lt4_dose=lt4,
            lt3_dose=lt3,
            rtf=rtf
        )

        means = compute_means(df)

        row = {
            "height": h,
            "weight": w,
            "sex": s,
            "lt4": lt4,
            "lt3": lt3,
            "RTF": rtf,
            "timeseries": df,
            **means
        }

        rows.append(row)

    dataset = pd.DataFrame(rows)

    return dataset


if __name__ == "__main__":
    dataset = generate_full_dataset()
    dataset.to_pickle("thyrosim_full_dataset.pkl")

    print("\nSaved dataset to thyrosim_full_dataset.pkl")

Total simulations: 64
[1/64] (1.56%)
[50/64] (78.12%)

Saved dataset to thyrosim_full_dataset.pkl


In [2]:
import os
os.chdir("..")

df = pd.read_pickle("thyrosim_full_dataset.pkl")

In [3]:
df_cut = df.drop(columns=["timeseries"])

df_cut.to_csv("simulation/thyrosim_cut_dataset.csv", index=False)

# for clearing memory (files too big)
import gc
del df
gc.collect()

0